# The sales project

importing

In [10]:
import pandas as pd
import sqlite3
import datetime
import logging
import datetime

write the logs to a file named pipeline.log
level = logging.INFO = sets the threshold on what it captures( WARNING, CRITICAL AND ERROR )

In [2]:

logging.basicConfig(filename='pipeline.log', level=logging.INFO, 
                    format='%(asctime)s - %(message)s')

logging.info('The data pipeline has started successfully.')

## Extraction

In [5]:
df = pd.read_csv('sales_raw.csv', encoding='latin1')
raw_count = len(df)
logging.info(f"Extract: {raw_count} records extracted from sales_raw.csv")

In [7]:
df.head()

,order_id,region,amount,date
0,CA-2016-152156,South,261.9600,11/8/2016
1,CA-2016-152156,South,731.9400,11/8/2016
2,CA-2016-138688,West,14.6200,6/12/2016
3,US-2015-108966,South,957.5775,10/11/2015
4,US-2015-108966,South,22.3680,10/11/2015


## Data Transform

In [6]:
cols_to_keep = {'Order ID': 'order_id', 'Region': 'region', 'Sales': 'amount', 'Order Date': 'date'}
df = df[list(cols_to_keep.keys())].rename(columns=cols_to_keep)

## Data Cleaning

In [9]:
df['amount'] = pd.to_numeric(df['amount'], errors = 'coerce') # Convert to numeric, set errors to NaN   
df = df.dropna(subset=['amount','order_id']) # Drop rows where 'amount' or 'order_id' is NaN
df['load_timestamp'] = datetime.datetime.now() # Add load timestamp

In [11]:
df = df[df['amount'] > 0] # Filter out records with non-positive sales amounts

In [19]:
dropped = raw_count - len(df)

if raw_count == final_count + dropped:
    status = "PASS"
    logging.info(f"Reconciliation: Success. {raw_count} (In) = {final_count} (Out) + {dropped} (Dropped).")
else:
    status = "FAIL"
    logging.error(f"Reconciliation: Mismatch! Lost {raw_count - final_count - dropped} rows.")

## Data Load

In [20]:
conn = sqlite3.connect('sales.db')
df.to_sql('sales_data', conn, if_exists='replace', index=False) # Load data into SQLite, replace existing data
final_count = pd.read_sql("SELECT COUNT(*) FROM sales_data", conn).iloc[0,0] # Get count of records in the database
logging.info(f"Load: {final_count} records loaded into sales.db")

In [21]:
report = pd.read_sql("""
            SELECT region, SUM(amount) as total_revenue, COUNT(order_id) as total_orders
            FROM sales_data 
            GROUP BY region
        """, conn)
        
report.to_csv('summary_report.csv', index=False)
        
conn.close()
print("Pipeline finished successfully! Check pipeline.log and summary_report.csv")

Pipeline finished successfully! Check pipeline.log and summary_report.csv


In [22]:
with open("summary.html", "w") as f:
    f.write(f"<h1>Pipeline Status: {status}</h1><p>Processed at: {datetime.datetime.now()}</p>")